<a href="https://colab.research.google.com/github/theritik08/Ritik-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theritik08/Ritik-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **scoring** task, not classification, clustering, or ranking in
the strict sense (though the output ends up as a ranked list).

I'm not sorting pages into fixed categories (that'd be classification), and
I'm not looking for natural groupings (that'd be clustering). What I'm
actually doing is predicting a continuous number — how much a page is
underperforming its expected CTR — for every page, then sorting by that
number. The ranking is just what you do with a score once you have it, not
the modeling task itself.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: predicted CTR for a page, given features knowable before anyone
reviewed it — position tier, average position, content type, main intent,
word count, age.

The proxy I actually care about is the gap between a page's real CTR and
what the model expected for a page like it: gap = actual_ctr -
predicted_ctr. A big negative gap means the page is underperforming what
similar pages achieve — that gap is my opportunity score.

This is a proxy, not an observed outcome — a low gap doesn't guarantee a
fixable problem, it just means "worth a look." There's no future-outcome
label here, it's a defined rule I'm building.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: R² (or MAE) of predicted CTR against a held-out set of
pages, compared to the tier-average baseline. "Good" means the model
explains meaningfully more variance in CTR than just knowing which of the
5 position tiers a page sits in. If it can't beat that simple baseline,
the extra complexity isn't worth it and the fixed rule should stay.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd, numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/theritik08/Ritik-Flyrank/refs/heads/main/data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one page
cols = ["content_id", "position_tier", "avg_position", "ctr",
        "impressions_90d", "content_type", "main_intent", "word_count"]

visible = df[df["impressions_90d"] >= 500].copy()

# Sketch of the target: expected CTR (tier mean, as a placeholder for a real model)
tier_mean = visible.groupby("position_tier")["ctr"].transform("mean")
visible["expected_ctr"] = tier_mean
visible["ctr_gap"] = visible["ctr"] - visible["expected_ctr"]

print("One row = one page. Sample sorted by biggest gap:")
visible[cols + ["expected_ctr", "ctr_gap"]].sort_values("ctr_gap").head(10)

One row = one page. Sample sorted by biggest gap:


,content_id,position_tier,avg_position,ctr,impressions_90d,content_type,main_intent,word_count,expected_ctr,ctr_gap
8855,content_eb72d6efc172,top_3,1.8,0.0,630,keyword article,informational,NaN,0.346572,-0.346572
21301,content_ee6c6b09c17d,top_3,2.8,0.0,916,keyword article,informational,1481.0,0.346572,-0.346572
11960,content_65c03925e49f,top_3,2.5,0.0,1041,keyword article,transactional,NaN,0.346572,-0.346572
2288,content_c3ccadb77bc4,top_3,2.4,0.0,741,keyword article,commercial,NaN,0.346572,-0.346572
26771,content_56c54a03ece2,top_3,2.7,0.0,1638,keyword article,informational,NaN,0.346572,-0.346572
26801,content_b03ea7190aea,top_3,2.6,0.0,1508,keyword article,commercial,2746.0,0.346572,-0.346572
22321,content_da01208f5c17,top_3,2.3,0.0,799,keyword article,transactional,3034.0,0.346572,-0.346572
29244,content_cca1e559cffd,top_3,2.9,0.0,1339,keyword article,commercial,1270.0,0.346572,-0.346572
24730,content_d0fd2e1ec8f6,top_3,2.5,0.0,519,keyword article,informational,5508.0,0.346572,-0.346572
18123,content_9e6c26757e7b,top_3,2.9,0.0,1135,keyword article,commercial,NaN,0.346572,-0.346572


Note: the top 10 by ctr_gap all show ctr=0.0 and an identical gap
(-0.347) — there are many zero-click top_3 pages tied at the bottom.
A real ranking would need a tiebreaker (e.g. sort ties by impressions_90d
descending) so the highest-exposure zero-click pages surface first. Also
noticed some word_count values are NaN — that'll need handling before
this feeds into an actual model.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The current fixed rule is "compare CTR to your position tier's average" —
5 buckets, one number each. That's already better than raw CTR, but it's
coarse: two pages both in "striking distance" (positions 4-10) get treated
identically, even though position 4 and position 10 have different real
CTR expectations.

A model can use the actual continuous avg_position instead of the bucket,
plus content_type and intent, which the tier rule ignores. That means it
can flag underperformance while accounting for exactly where a page ranks
and what kind of content it is — something a 5-bucket rule structurally
can't do. If the model can't beat the tier-mean baseline once I build it,
that's a real finding too — it'd mean the extra complexity isn't earning
its keep, and the simple rule was fine all along.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.